In [1]:
import os
import json
import random
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

GENRES = [
    "Drama", "Romance", "Comedy", "Sci-Fi",
    "Family", "Workplace", "Sports", "Psychological"
]

THEMES = [
    "second chances",
    "identity",
    "career transition",
    "forgiveness",
    "friendship",
    "ambition",
    "artificial intelligence",
    "small-town life"
]

BASE_PROMPT = """
You are a professional screenwriter.

Write a modern screenplay.

Constraints:
- No guns
- No gun violence
- No weapon combat
- No graphic violence
- No copyrighted characters
- Completely original story

Requirements:
- 10–15 scenes
- Proper screenplay format
- INT/EXT scene headers
- Dialogue in uppercase character format
- Realistic pacing

Genre: {genre}
Theme: {theme}

Begin screenplay.
"""

def generate_script():
    genre = random.choice(GENRES)
    theme = random.choice(THEMES)

    prompt = BASE_PROMPT.format(genre=genre, theme=theme)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=2500,
        temperature=0.9,
        top_p=0.95,
        do_sample=True
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {
        "genre": genre,
        "theme": theme,
        "script": text
    }

OUTPUT_DIR = "datasets/synthetic"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#for i in tqdm(range(1000)):
for i in tqdm(range(850, 1000)):
    script_data = generate_script()

    with open(f"{OUTPUT_DIR}/script_{i:04d}.json", "w", encoding="utf-8") as f:
        json.dump(script_data, f, ensure_ascii=False, indent=2)

/home/harris/miniconda3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████| 291/291 [00:02<00:00, 101.82it/s, Materializing param=model.norm.weight]
Some parameters are on the meta device because they were offloaded to the cpu.
100%|██████████████████████████████████████████████████████████████████████████████| 150/150 [5:59:59<00:00, 144.00s/it]
